# Boolean network sampling — Case Study 1

Generate **50 Boolean networks by default** from the selected optimal regulatory graph. Change `N_MODELS` if you need another number.

All generated files are saved in **`../Results/Boolean_network_sampling_outputs/`**.

**Inputs:** the optimal graph and gene list from component analysis, the selection manifest, the binarized S0–S5 observations, and the mouse gene ID reference. Run the component-analysis notebook first.

## 1. Settings and input files

In [1]:
from pathlib import Path
import hashlib
import json
import time

import pandas as pd

# Sampling settings
N_MODELS = 50
DIVERSE = True
MAXCLAUSE = 8

# Input files from the completed component and binarization analyses
COMPONENT_DIR = Path("../Results/Component_analysis_outputs")
MODEL_SIF = COMPONENT_DIR / "synthesis_core_network.sif"
MODEL_GENES_TXT = COMPONENT_DIR / "core_genes.txt"
SELECTION_MANIFEST = COMPONENT_DIR / "selection_manifest.json"
OBS_CSV = Path("../Data/nestorowa_singlecell_binarized_observations.csv")
NCBI_INFO = Path("../Data/Mus_musculus.gene_info.202103.tsv")

# All generated files go here
OUTPUT_DIR = Path("../Results/Boolean_network_sampling_outputs")

if not isinstance(N_MODELS, int) or N_MODELS < 1:
    raise ValueError("N_MODELS must be a positive integer.")
for path in (MODEL_SIF, MODEL_GENES_TXT, SELECTION_MANIFEST, OBS_CSV, NCBI_INFO):
    if not path.is_file():
        raise FileNotFoundError(f"Missing required input: {path}")

# Protect prior results before writing any new output.
if list(OUTPUT_DIR.glob("bn*.bnet")) or (OUTPUT_DIR / "model_manifest.csv").exists():
    raise FileExistsError(
        f"Previous sampling results exist in {OUTPUT_DIR}. "
        "Move or archive them before starting another run."
    )

print(f"Requested models: {N_MODELS}")
print(f"Sampling: {'diverse' if DIVERSE else 'ordinary'}")
print(f"Output directory: {OUTPUT_DIR}")

Requested models: 50
Sampling: diverse
Output directory: ../Results/Boolean_network_sampling_outputs


## 2. Load the optimal regulatory graph

Read the core genes and their signed interactions. Check that the graph matches the exported gene list and came from a **proven optimal** component-selection result.

In [8]:
edges = pd.read_csv(
    MODEL_SIF, sep="\t", header=None,
    names=["source", "sign", "target"], dtype=str
)
if edges.empty or edges.isna().any().any():
    raise ValueError("The input signed graph is empty or invalid.")

for column in ("source", "sign", "target"):
    edges[column] = edges[column].str.strip()

modeled_nodes = set(MODEL_GENES_TXT.read_text().splitlines()) - {""}
graph_nodes = set(edges["source"]) | set(edges["target"])
if modeled_nodes != graph_nodes:
    raise ValueError(
        "Graph nodes do not match the gene list. "
        f"Missing from graph: {sorted(modeled_nodes - graph_nodes)}; "
        f"extra in graph: {sorted(graph_nodes - modeled_nodes)}"
    )

selection = json.loads(SELECTION_MANIFEST.read_text())
if not selection.get("clingo_optimum_proven", False):
    raise RuntimeError("Component selection has no proven global optimum.")
if len(modeled_nodes) != selection["n_core_genes"]:
    raise ValueError("Core gene count differs from the optimal selection manifest.")

sif_hash = hashlib.sha256(MODEL_SIF.read_bytes()).hexdigest()
expected_hash = selection.get("core_sif_sha256")
if expected_hash and sif_hash != expected_hash:
    raise ValueError("The network file differs from the selected optimal graph.")

print("Genes:", len(modeled_nodes), "Signed edges:", len(edges))

Genes: 39 Signed edges: 157


## 3. Load the binarized cell-state observations

Convert the S0–S5 observations to Boolean values: **0 = OFF**, **1 = ON**, and **ND / missing = unknown**. Unknown entries are not used as constraints. Stop if a state has no determined genes, because that would make its observation unconstrained.

In [9]:
import gene_name_standardization as gns

raw_observations = pd.read_csv(OBS_CSV, index_col=0).to_dict(orient="index")
standardized = gns.observations_standardization(raw_observations, str(NCBI_INFO))
states = ("S0", "S1", "S2", "S3", "S4", "S5")
if not set(states).issubset(standardized):
    raise ValueError(f"Missing states: {sorted(set(states) - set(standardized))}")

# Match gene identifiers without changing the names used in the network.
node_names = {gene.upper(): gene for gene in modeled_nodes}
if len(node_names) != len(modeled_nodes):
    raise ValueError("Core gene symbols are ambiguous after case normalization.")


def binary_value(value):
    """Convert a 0/1/ND observation to 0, 1, or None."""
    if pd.isna(value):
        return None
    if isinstance(value, str):
        value = value.strip().upper()
        if value in ("", "ND", "NA", "NAN", "NONE", "?"):
            return None
    try:
        numeric = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Unexpected observation value: {value!r}") from exc
    if numeric in (0.0, 1.0):
        return int(numeric)
    raise ValueError(f"Unexpected observation value: {value!r}. Expected 0, 1, or ND.")


observations = {}
for state in states:
    observations[state] = {}
    for gene, raw_value in standardized[state].items():
        matching_gene = node_names.get(str(gene).strip().upper())
        if matching_gene is None:
            continue
        value = binary_value(raw_value)
        if value is not None:
            if matching_gene in observations[state] and observations[state][matching_gene] != value:
                raise ValueError(f"Conflicting values for {matching_gene} in {state}.")
            observations[state][matching_gene] = value

counts = {state: len(observations[state]) for state in states}
print("Determined observations per state:", counts)
if any(count == 0 for count in counts.values()):
    raise RuntimeError(
        "At least one state has no determined genes. Check gene-name mapping "
        "and binarization before sampling."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame.from_dict(observations, orient="index").rename_axis("state").to_csv(
    OUTPUT_DIR / "observations_used.csv", na_rep="ND"
)

Determined observations per state: {'S0': 12, 'S1': 11, 'S2': 12, 'S3': 10, 'S4': 10, 'S5': 11}


## 4. Define all BoNesis constraints

This is the full original specification: three distinct fixed points (S2, S4, S5), five required reachability links, and one forbidden reachability link. The selected gene set stays fixed. No constraints are removed for sampling.

In [10]:
import bonesis

grn = bonesis.InfluenceGraph.from_sif(
    str(MODEL_SIF), maxclause=MAXCLAUSE,
    allow_skipping_nodes=False, canonic=True
)
if set(grn.nodes()) != modeled_nodes:
    raise ValueError("BoNesis changed the set of genes in the graph.")

bo = bonesis.BoNesis(grn, observations)

# The three terminal observations must be distinct fixed points.
s2 = bo.fixed(~bo.obs("S2"))
s4 = bo.fixed(~bo.obs("S4"))
s5 = bo.fixed(~bo.obs("S5"))
s2 != s4
s2 != s5
s4 != s5

# Required differentiation trajectories.
~bo.obs("S1") >= ~bo.obs("S0") >= s2
~bo.obs("S0") >= ~bo.obs("S3") >= s4
~bo.obs("S3") >= s5

# Forbidden trajectory.
~bo.obs("S3") / s2

print("Complete dynamical constraints registered with BoNesis.")
print("Candidate genes:", len(grn.nodes()), "candidate edges:", len(grn.edges()))

ipylab module is not installed, menus and toolbar are disabled.


Complete dynamical constraints registered with BoNesis.
Candidate genes: 39 candidate edges: 157


## 5. Generate and save the requested Boolean networks

BoNesis returns networks that satisfy **the full specification above**. Every generated network is checked to have the expected genes and a unique saved file. The notebook treats the run as **complete only if exactly `N_MODELS` networks were obtained**.

The model files, manifest, and run metadata are saved in `../Results/Boolean_network_sampling_outputs/`. If the solver is interrupted or returns fewer models, the partial outputs remain for diagnosis, and the run is explicitly marked incomplete. A request for 50 models cannot guarantee that the solver will find 50 within a particular time.

In [ ]:
view = (
    bo.diverse_boolean_networks(limit=N_MODELS)
    if DIVERSE else bo.boolean_networks(limit=N_MODELS)
)

manifest = []
seen_hashes = set()
start = time.perf_counter()

try:
    for index, bn in enumerate(view):
        # Each model must include exactly the selected genes.
        if set(bn) != modeled_nodes:
            raise RuntimeError(f"Model {index} has an unexpected gene set.")

        path = OUTPUT_DIR / f"bn{index}.bnet"
        bn.save(str(path))
        digest = hashlib.sha256(path.read_bytes()).hexdigest()

        if digest in seen_hashes:
            path.unlink()
            raise RuntimeError(f"Duplicate network encountered at model {index + 1}.")
        seen_hashes.add(digest)

        manifest.append({
            "model_id": index,
            "filename": path.name,
            "sha256": digest,
            "size_bytes": path.stat().st_size,
        })
        print(f"Saved {len(manifest)}/{N_MODELS} networks", flush=True)

finally:
    # Save what happened even if sampling was interrupted.
    completed = len(manifest) == N_MODELS
    metadata = {
        "status": "complete" if completed else "incomplete",
        "n_requested": N_MODELS,
        "n_saved": len(manifest),
        "diverse_sampling": DIVERSE,
        "maxclause": MAXCLAUSE,
        "n_genes": len(modeled_nodes),
        "n_candidate_edges": len(edges),
        "optimal_source_answer_id": selection["selected_answer_id"],
        "source_proven_optimal": True,
        "input_sif_sha256": sif_hash,
        "constraint_semantics": "BoNesis Most Permissive",
        "constraint_set": [
            "S2, S4, S5: pairwise distinct fixed points",
            "S1 -> S0 -> S2",
            "S0 -> S3 -> S4",
            "S3 -> S5",
            "S3 cannot reach S2",
        ],
        "elapsed_seconds": round(time.perf_counter() - start, 1),
    }
    pd.DataFrame(
        manifest, columns=["model_id", "filename", "sha256", "size_bytes"]
    ).to_csv(OUTPUT_DIR / "model_manifest.csv", index=False)
    (OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2))

if len(manifest) != N_MODELS:
    raise RuntimeError(
        f"INCOMPLETE: only {len(manifest)} of {N_MODELS} requested networks were generated. "
        "See run_metadata.json."
    )

print(f"COMPLETE: {N_MODELS} distinct, BoNesis-constrained networks saved to {OUTPUT_DIR}")

## 6. Attractor Check


In [13]:
from pathlib import Path
import pandas as pd
import mpbn

# Directory containing the generated Boolean networks
MODELS_DIR = Path("../Results/Boolean_network_sampling_outputs")

# Find all generated models
model_files = sorted(
    MODELS_DIR.glob("bn*.bnet"),
    key=lambda p: int(p.stem[2:])
)

print(f"Found {len(model_files)} Boolean networks")

results = []

for file in model_files:

    # Load the Boolean network
    bn = mpbn.MPBooleanNetwork(str(file))

    # Compute all Most Permissive attractors
    attractors = list(bn.attractors())

    # Non-fixed attractors contain at least one "*" value
    nonfixed = [
        a for a in attractors
        if "*" in a.values()
    ]

    # Record the results
    results.append({
        "model": file.name,
        "n_attractors": len(attractors),
        "n_fixed_points": len(attractors) - len(nonfixed),
        "n_nonfixed_attractors": len(nonfixed),
        "has_nonfixed_attractor": len(nonfixed) > 0
    })

    print(
        f"{file.name}: "
        f"{len(attractors)} attractors, "
        f"{len(nonfixed)} non-fixed"
    )

# Create the results table
results_df = pd.DataFrame(results)

# Save the results
results_df.to_csv(
    MODELS_DIR / "attractor_analysis.csv",
    index=False
)

display(results_df)

Found 50 Boolean networks
bn0.bnet: 6 attractors, 0 non-fixed
bn1.bnet: 8 attractors, 0 non-fixed
bn2.bnet: 8 attractors, 0 non-fixed
bn3.bnet: 16 attractors, 0 non-fixed
bn4.bnet: 9 attractors, 0 non-fixed
bn5.bnet: 10 attractors, 0 non-fixed
bn6.bnet: 6 attractors, 0 non-fixed
bn7.bnet: 8 attractors, 0 non-fixed
bn8.bnet: 16 attractors, 0 non-fixed
bn9.bnet: 6 attractors, 0 non-fixed
bn10.bnet: 6 attractors, 0 non-fixed
bn11.bnet: 5 attractors, 0 non-fixed
bn12.bnet: 10 attractors, 0 non-fixed
bn13.bnet: 10 attractors, 0 non-fixed
bn14.bnet: 8 attractors, 0 non-fixed
bn15.bnet: 6 attractors, 0 non-fixed
bn16.bnet: 5 attractors, 0 non-fixed
bn17.bnet: 16 attractors, 0 non-fixed
bn18.bnet: 32 attractors, 0 non-fixed
bn19.bnet: 5 attractors, 0 non-fixed
bn20.bnet: 3 attractors, 0 non-fixed
bn21.bnet: 10 attractors, 0 non-fixed
bn22.bnet: 3 attractors, 0 non-fixed
bn23.bnet: 4 attractors, 0 non-fixed
bn24.bnet: 8 attractors, 0 non-fixed
bn25.bnet: 6 attractors, 0 non-fixed
bn26.bnet: 10 

,model,n_attractors,n_fixed_points,n_nonfixed_attractors,has_nonfixed_attractor
0,bn0.bnet,6,6,0,False
1,bn1.bnet,8,8,0,False
2,bn2.bnet,8,8,0,False
3,bn3.bnet,16,16,0,False
4,bn4.bnet,9,9,0,False
5,bn5.bnet,10,10,0,False
6,bn6.bnet,6,6,0,False
7,bn7.bnet,8,8,0,False
8,bn8.bnet,16,16,0,False
9,bn9.bnet,6,6,0,False
